# Moteur de recherche d'articles scientifiques — Elements Impact

### Pipeline

#### Architecture du pipeline

```
Titre + Description du projet + Groupe ou Groupes bénéficiaires du projet + tags
            │
            ▼
┌─────────────────────────┐
│  ÉTAPE 1 — LLM          │  Sélection des prédicteurs pertinents
│  (Claude / GPT)         │  parmi les 79 du modèle Margolis
└──────────┬──────────────┘
           │  Liste de prédicteurs
           ▼
┌─────────────────────────┐
│  ÉTAPE 2 — OpenAlex     │  Recherche d'articles scientifiques
│  (API gratuite)         │  par prédicteur + contexte projet
└──────────┬──────────────┘
           │  Articles (titre, abstract, DOI)
           ▼
┌─────────────────────────┐
│  ÉTAPE 3 — LLM          │  Extraction de l'effect size
│  (extraction NLP)       │  Cohen's d / Hedge's g + durée
└──────────┬──────────────┘
           │
           ▼
   DataFrame structuré
   prêt pour Boussole

## Configurations

In [ ]:
# Configurations

import os, json, time, requests
import pandas as pd
from IPython.display import display, Markdown

# Clé API Anthropic
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")

# Email pour obtenires requetes prioritaires
OPENALEX_EMAIL = "ksisaber@gmail.com"

# Fichier des prédicteurs Margolis utilisé par Elements
PREDICTORS_FILE = "articles_dataset.xlsx"

# Paramètres du moteur
TOP_N_PREDICTORS       = 2    # Nombre de prédicteurs à explorer par projet
ARTICLES_PER_PREDICTOR = 4    # Nombre d'articles retenus par prédicteur (après filtrage)
MIN_CITATIONS          = 3    # Filtre qualité : citations minimum par article
OPENALEX_FETCH         = 30  # Articles récupérés depuis OpenAlex avant sélection du top N


## Entrées du projet

In [7]:
# Inputs du projet

PROJECT = {

    # Nom court du projet
    "title": "Early childhood support and parenting center",

    # Description des actions menées dans le projet
    "description": """
        Management of a welcoming space dedicated to parenting support
        and early childhood development for children in their first years of life.
        Weekly workshops, individual coaching sessions, peer support groups,
        and access to child development resources.
    """,

    # Groupe(s) bénéficiaire(s)
    "target_group": "Children aged 0-6 and their parents",

    # Tags libres — optionnel, améliore la pertinence des requêtes
    "tags": ["early childhood", "parenting", "child development", "social support"],
    
    "search_context": "parenting support",
}


## Moteur de recherche

In [8]:
# Moteur de recherche

# Chargement des prédicteurs

df_pred = pd.read_excel(PREDICTORS_FILE, sheet_name="Predictors")
df_pred = df_pred[["Predictor (EN)", "Predictor (FR)", "Domain (FR)"]].dropna(subset=["Predictor (EN)"])
df_pred.columns = ["predictor_en", "predictor_fr", "domain_fr"]

PREDICTORS_FOR_PROMPT = "\n".join(
    f"- [{row.domain_fr}] {row.predictor_en}"
    for _, row in df_pred.iterrows()
)


# Helpers

def call_claude(prompt: str, system: str = "", max_tokens: int = 2000) -> str:
    """Appelle Claude et retourne le texte de la réponse."""
    resp = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={
            "x-api-key": ANTHROPIC_API_KEY,
            "anthropic-version": "2023-06-01",
            "content-type": "application/json",
        },
        json={
            "model": "claude-sonnet-4-20250514",
            "max_tokens": max_tokens,
            "system": system,
            "messages": [{"role": "user", "content": prompt}],
        },
        timeout=60
    )
    resp.raise_for_status()
    return resp.json()["content"][0]["text"]


def parse_json(text: str):
    """Extrait et parse le JSON d'une réponse LLM."""
    if "```json" in text:
        text = text.split("```json")[1].split("```")[0]
    elif "```" in text:
        text = text.split("```")[1].split("```")[0]
    return json.loads(text.strip())


def reconstruct_abstract(idx: dict | None) -> str:
    """
    Reconstitue l'abstract depuis le format index inversé d'OpenAlex.
    OpenAlex stocke les abstracts sous la forme {mot: [positions]} pour
    des raisons de droit d'auteur. Cette fonction reconstitue le texte lisible.
    """
    if not idx:
        return ""
    slots = [""] * (max(p for pp in idx.values() for p in pp) + 1)
    for word, positions in idx.items():
        for pos in positions:
            slots[pos] = word
    return " ".join(slots)

### Étape 1 : Sélection des prédicteurs pertinents 

On utilise un LLM qui reçoit les 79 prédicteurs et sélectionne ceux que le projet est susceptible d'impacter avec un score de pertinence de 1 a 5

In [9]:
print("🔎 Étape 1/3 — Identification des prédicteurs pertinents...")

# Ce prompt guide le LLM à sélectionner les prédicteurs les plus pertinents pour le projet donné
resp_predictors = call_claude(
    system="Tu es expert en évaluation d'impact social et modèle Margolis SWB. Réponds UNIQUEMENT en JSON valide.",
    prompt=f"""
Projet : {PROJECT['title']}
Groupe : {PROJECT['target_group']}
Description : {PROJECT['description']}
Tags : {', '.join(PROJECT['tags'])}

Voici les 79 prédicteurs du modèle Margolis :
{PREDICTORS_FOR_PROMPT}

Sélectionne les {TOP_N_PREDICTORS} prédicteurs les plus susceptibles d'être impactés.
Pour chacun :
- predictor_en    : nom exact du prédicteur
- domain_fr       : domaine
- relevance_score : 1-5
- justification   : 1 phrase

JSON uniquement :
```json
[{{"predictor_en": "...", "domain_fr": "...", "relevance_score": 5, "justification": "..."}}]
```""",
    max_tokens=1500
)

selected = sorted(parse_json(resp_predictors), key=lambda x: x.get("relevance_score", 0), reverse=True)
print(f"   → {len(selected)} prédicteurs identifiés")

🔎 Étape 1/3 — Identification des prédicteurs pertinents...
   → 2 prédicteurs identifiés


### Étapes 2 & 3 : Recherche OpenAlex + Extraction effect size

Pour chaque prédicteur, on :
1. Génère **jusqu'à 10 requêtes 100% déterministes** avec contexte projet intégré (terme anchor)
2. Récupère le **top 5 par requête** depuis OpenAlex → pool ~50 articles dédupliqués par DOI
3. Le **LLM sélectionne les N meilleurs** selon pertinence projet + prédicteur + présence probable d'effect size
4. Extrait le Cohen's d / Hedge's g + **justification de sélection** depuis les abstracts

In [10]:
results   = []
seen_dois = set()  # déduplication globale inter-prédicteurs

# ── Constantes ────────────────────────────────────────────────────────────────
OPENALEX_FETCH = 20   # articles récupérés par requête (on prend ensuite top TOP_PER_QUERY)
TOP_PER_QUERY  = 5    # top N articles retenus par requête pour constituer le pool LLM

# ── Helpers ───────────────────────────────────────────────────────────────────

def _unique_preserve_order(lst):
    seen = set()
    return [x for x in lst if not (x in seen or seen.add(x))]


def build_queries(pred_name, project):
    """
    Génère jusqu'à 10 requêtes déterministes.

    Si project["search_context"] est renseigné (ex: "parenting support"),
    ces termes sont injectés dans les requêtes Q6-Q10 (sans meta-analysis)
    pour ancrer la recherche dans le domaine spécifique du projet.
    Les requêtes Q1-Q5 (avec meta-analysis) restent génériques pour maximiser
    le recall sur les méta-analyses disponibles.

    Sans search_context, toutes les requêtes utilisent les termes extraits
    automatiquement du titre, groupe et tags.
    """
    import re as _re

    def tokenize(text):
        stopwords = {
            "a","an","the","of","in","for","on","with","to","and","or","at","by","as","is","are",
            "was","were","be","been","being","have","has","had","do","does","did","will","would",
            "could","should","may","might","its","it","this","that","these","those","their","our",
            "your","his","her","un","une","le","la","les","de","du","des","en","et","ou","au","aux",
            "pour","par","sur","dans","avec","qui","que","dont",
            "center","support","program","programme","project","service","based","early","young",
        }
        words = _re.findall(r"[a-zA-Zéèêàùîôûç]{3,}", text.lower())
        return [w for w in words if w not in stopwords]

    pred    = tokenize(pred_name)
    ctx     = tokenize(project.get("title", ""))
    desc    = tokenize(project.get("description", ""))[:8]
    group   = tokenize(project.get("target_group", ""))
    tags    = tokenize(" ".join(project.get("tags", [])))

    # Contexte de recherche saisi par l'utilisateur
    search_ctx_raw   = project.get("search_context", "").strip()
    search_ctx_terms = tokenize(search_ctx_raw) if search_ctx_raw else []
    search_ctx       = " ".join(search_ctx_terms[:3])

    queries = []

    # ── Groupe 1 : avec meta-analysis / systematic review ─────────────────────
    # Requêtes génériques — recall maximal sur les méta-analyses

    # Q1 : prédicteur seul + search context + meta-analysis
    if pred:
        queries.append(" ".join(pred[:4]) + f" {search_ctx}" + " meta-analysis")
    # Q2 : prédicteur + groupe cible + meta-analysis
    if pred and group:
        queries.append(" ".join(pred[:3] + group[:2]) + " meta-analysis")
    # Q3 : prédicteur + tags + meta-analysis
    if pred and tags:
        queries.append(" ".join(pred[:3] + tags[:2]) + " meta-analysis")
    # Q4 : prédicteur + contexte titre + systematic review
    if pred and ctx:
        queries.append(" ".join(pred[:3] + ctx[:2]) + " systematic review")
    # Q5 : prédicteur + wellbeing + systematic review (ancrage SWB Margolis)
    if pred:
        queries.append(" ".join(pred[:3]) + " wellbeing systematic review")

    # ── Groupe 2 : sans meta-analysis — search_context injecté ici ────────────

    # Q6 : prédicteur + search_context + wellbeing
    if pred and search_ctx:
        queries.append(" ".join(pred[:3]) + f" {search_ctx} wellbeing")
    elif pred and group:
        queries.append(" ".join(pred[:3] + group[:2]) + " wellbeing")

    # Q7 : prédicteur + search_context + intervention
    if pred and search_ctx:
        queries.append(" ".join(pred[:3]) + f" {search_ctx} intervention")
    elif pred and group:
        queries.append(" ".join(pred[:3]) + " intervention " + " ".join(group[:2]))

    # Q8 : prédicteur + search_context seul
    if pred and search_ctx:
        queries.append(" ".join(pred[:3]) + f" {search_ctx}")
    elif pred and ctx:
        queries.append(" ".join(pred[:3] + ctx[:3]))

    # Q9 : prédicteur + search_context + groupe
    if pred and search_ctx and group:
        queries.append(" ".join(pred[:2]) + f" {search_ctx} " + " ".join(group[:2]))
    elif pred and tags:
        queries.append(" ".join(pred[:3] + tags[:3]))

    # Q10 : combinaison maximale
    if search_ctx:
        all_terms = _unique_preserve_order(pred[:2] + search_ctx_terms[:2] + group[:2])
    else:
        all_terms = _unique_preserve_order(pred[:2] + ctx[:2] + group[:2] + tags[:2])
    if all_terms:
        queries.append(" ".join(all_terms[:9]))

    return _unique_preserve_order([q.strip() for q in queries if q.strip()])[:10]


def fetch_openalex(q):
    """Récupère OPENALEX_FETCH articles, filtre citations, retourne top TOP_PER_QUERY."""
    oa_resp = requests.get(
        "https://api.openalex.org/works",
        params={
            "search"  : q,
            "per-page": OPENALEX_FETCH,
            "sort"    : "relevance_score:desc",
            "select"  : "title,abstract_inverted_index,doi,publication_year,cited_by_count,open_access,authorships",
            "mailto"  : OPENALEX_EMAIL,
        },
        timeout=30
    ).json().get("results", [])
    articles = [
        {
            "title"   : w.get("title", ""),
            "abstract": reconstruct_abstract(w.get("abstract_inverted_index")),
            "doi"     : w.get("doi", ""),
            "year"    : w.get("publication_year"),
            "cited"   : w.get("cited_by_count", 0),
            "oa"      : w.get("open_access", {}).get("is_oa", False),
        }
        for w in oa_resp
        if w.get("cited_by_count", 0) >= MIN_CITATIONS
    ]
    return articles[:TOP_PER_QUERY]


def select_articles(pool, pred_name, project, n):
    """
    Le LLM reçoit tous les candidats du pool et sélectionne les N meilleurs.
    Critères : compatibilité contexte/population projet + lien prédicteur
    + présence probable d'un effect size quantitatif.
    Appel unique, max_tokens minimal (juste une liste d'indices).
    """
    if not pool or len(pool) <= n:
        return pool

    candidates = "\n".join(
        f"{i+1}. [{art.get('cited',0)} cit.] {art['title']} ({art.get('year','')})"
        f" — {(art.get('abstract','') or '')[:180]}"
        for i, art in enumerate(pool)
    )

    raw = call_claude(
        system="You are an expert in social impact evaluation. Be strict and concise.",
        prompt=f"""Project: {project['title']}
Description: {project.get('description','')[:300]}
Target group: {project['target_group']}
Tags: {", ".join(project.get('tags',[]))}
Predictor studied: {pred_name}

Candidate articles ({len(pool)} total):
{candidates}

Select the {n} best articles. Prioritize:
1. Population/context compatible with the project and target group
2. Directly measuring or studying "{pred_name}"
3. Likely to contain a quantitative effect size (meta-analysis, RCT, systematic review)

Return ONLY a JSON list of {n} integers (1-based). Example: [3, 7, 12]""",
        max_tokens=60
    )

    try:
        match = re.search(r'\[.*?\]', raw, re.DOTALL)
        indices = json.loads(match.group()) if match else []
        selected = [pool[i-1] for i in indices if isinstance(i, int) and 1 <= i <= len(pool)]
        # Fallback si LLM retourne moins que demandé
        if len(selected) < n:
            doi_sel = {a.get('doi') for a in selected}
            for art in pool:
                if len(selected) >= n: break
                if art.get('doi') not in doi_sel:
                    selected.append(art)
        return selected[:n]
    except Exception:
        return pool[:n]


# ── Boucle principale ─────────────────────────────────────────────────────────

for i, pred in enumerate(selected):
    p_name = pred["predictor_en"]
    print(f"\n📄 [{i+1}/{len(selected)}] {p_name}")

    # Génération des requêtes avec contexte anchor
    queries = build_queries(p_name, PROJECT)
    if not queries:
        queries = [f"{p_name} intervention effect size meta-analysis"]

    n_meta = sum(1 for q in queries if "meta-analysis" in q or "systematic" in q)
    print(f"   📋 {len(queries)} requêtes ({n_meta} avec meta-analysis/systematic review)")
    for q in queries:
        marker = "🔬" if "meta-analysis" in q or "systematic" in q else "📖"
        print(f"   {marker} {q}")

    # ── Étape 2a : top TOP_PER_QUERY par requête → pool dédupliqué ─────────────
    seen_local = set()
    pool = []
    for q in queries:
        for art in fetch_openalex(q):
            doi = art.get("doi")
            if doi and doi in seen_local:
                continue
            if doi:
                seen_local.add(doi)
            pool.append(art)

    if not pool:
        fallback = f"{p_name} intervention effect size meta-analysis"
        print(f"   ⚠️  0 résultats → requête de secours : '{fallback}'")
        pool = fetch_openalex(fallback)

    print(f"   📥 Pool : {len(pool)} articles (top {TOP_PER_QUERY}/requête, dédupliqués)")

    # ── Étape 2b : sélection LLM des N meilleurs du pool ──────────────────────
    print(f"   🤖 Sélection LLM des {ARTICLES_PER_PREDICTOR} meilleurs parmi {len(pool)}...")
    try:
        selected_pool = select_articles(pool, p_name, PROJECT, ARTICLES_PER_PREDICTOR)
    except Exception as e_sel:
        print(f"   ⚠️  Sélection LLM échouée ({e_sel}) → fallback citations")
        selected_pool = sorted(pool, key=lambda x: x.get("cited", 0), reverse=True)[:ARTICLES_PER_PREDICTOR]

    # ── Déduplication DOI inter-prédicteurs ────────────────────────────────────
    articles = []
    for art in selected_pool:
        doi = art.get("doi")
        if doi and doi in seen_dois: continue
        if doi: seen_dois.add(doi)
        articles.append(art)

    print(f"   ✅ {len(articles)} article(s) retenus pour extraction")

    # ── Étape 3 : extraction effect size ──────────────────────────────────────
    for art in articles:
        abstract = art["abstract"][:2000] if art["abstract"] else "[Non disponible]"

        ext = call_claude(
            system="Tu es expert en méta-analyse. Extrais les tailles d'effet des abstracts. Réponds UNIQUEMENT en JSON valide.",
            prompt=f"""Article : {art['title']} ({art['year']})
Abstract : {abstract}

Prédicteur ciblé : {p_name}
Projet : {PROJECT['title']} | Groupe : {PROJECT['target_group']}

Retiens l'effect size le plus directement lié à {p_name}.

```json
{{
  "effect_size"      : <float ou null>,
  "effect_type"      : <"Cohen's d"|"Hedge's g"|"SMD"|"autre"|null>,
  "effect_direction" : <"positif"|"négatif"|null>,
  "effect_duration"  : <durée de l'effet, ex: "6 months", "1 year" — null si non mentionné>,
  "relevance"        : <1-5>,
  "confidence"       : <1-5>,
  "source_text"      : <citation exacte du passage d'où l'effect size a été extrait, ou null>,
  "selection_reason" : <1 phrase expliquant pourquoi cet article est pertinent pour "{p_name}" dans ce projet — ou null>,
  "note"             : <string court>
}}
```""",
            max_tokens=600
        )

        try:
            parsed = parse_json(ext)
        except Exception:
            parsed = {}

        es = parsed.get("effect_size")
        if es is not None:
            print(f"   ✅ {art['title'][:55]}… → {es}")
        else:
            print(f"   ⚠️  {art['title'][:55]}… → effect size non trouvé")

        results.append({
            "predictor"       : p_name,
            "domain"          : pred["domain_fr"],
            "pred_relevance"  : pred["relevance_score"],
            "pred_justif"     : pred["justification"],
            "search_queries"  : " | ".join(queries),
            "title"           : art["title"],
            "year"            : art["year"],
            "doi"             : art["doi"],
            "cited_by"        : art["cited"],
            "open_access"     : art["oa"],
            "effect_size"     : parsed.get("effect_size"),
            "effect_type"     : parsed.get("effect_type"),
            "effect_direction": parsed.get("effect_direction"),
            "effect_duration" : parsed.get("effect_duration"),
            "art_relevance"   : parsed.get("relevance"),
            "confidence"      : parsed.get("confidence"),
            "source_text"     : parsed.get("source_text", ""),
            "selection_reason": parsed.get("selection_reason", ""),
            "note"            : parsed.get("note", ""),
        })
        time.sleep(0.05)

print("\n✅ Recherche terminée")



📄 [1/2] Family support
   📋 9 requêtes (5 avec meta-analysis/systematic review)
   🔬 family parenting meta-analysis
   🔬 family children aged meta-analysis
   🔬 family childhood parenting meta-analysis
   🔬 family childhood parenting systematic review
   🔬 family wellbeing systematic review
   📖 family parenting wellbeing
   📖 family parenting intervention
   📖 family parenting
   📖 family parenting children aged
   📥 Pool : 43 articles (top 5/requête, dédupliqués)
   🤖 Sélection LLM des 4 meilleurs parmi 43...
   ✅ 4 article(s) retenus pour extraction
   ⚠️  Risk of Mental Illness in Offspring of Parents With Sch… → effect size non trouvé
   ⚠️  A meta-analysis update on the effects of early family/p… → effect size non trouvé
   ⚠️  A meta-analysis on interparental conflict, parenting, a… → effect size non trouvé
   ⚠️  Do the Parent–Child Relationship and Parenting Behavior… → effect size non trouvé

📄 [2/2] Social integration
   📋 9 requêtes (5 avec meta-analysis/systematic review)

## Résultats

In [11]:
# Résultats

df = pd.DataFrame(results)

# ── Résumé ───────────────────────────────────────────────────────────────────
n_total       = len(df)
n_with_effect = df["effect_size"].notna().sum()
n_predictors  = df["predictor"].nunique()
top_articles  = df[df["effect_size"].notna()].sort_values(
    ["art_relevance", "confidence", "cited_by"], ascending=False
)

display(Markdown(f"""
## 📊 Résultats — {PROJECT['title']}
**Groupe cible :** {PROJECT['target_group']}

| Métrique | Valeur |
|---|---|
| Prédicteurs explorés | {n_predictors} |
| Articles analysés | {n_total} |
| Articles avec effect size | {n_with_effect} ({n_with_effect/n_total:.0%} si >0) |
"""))

# ── Prédicteurs sélectionnés ─────────────────────────────────────────────────
display(Markdown("### Prédicteurs identifiés"))
display(
    pd.DataFrame(selected)[["predictor_en", "domain_fr", "relevance_score", "justification"]]
    .rename(columns={
        "predictor_en"  : "Prédicteur",
        "domain_fr"     : "Domaine",
        "relevance_score": "Score",
        "justification" : "Justification"
    })
)

# ── Articles avec effect size ─────────────────────────────────────────────────
display(Markdown("### 📄 Articles avec effect size extrait"))

if not top_articles.empty:
    display(
        top_articles[[
            "predictor", "title", "year", "cited_by",
            "effect_size", "effect_type", "effect_duration",
            "art_relevance", "confidence", "note", "doi"
        ]].rename(columns={
            "predictor"      : "Prédicteur",
            "title"          : "Titre",
            "year"           : "Année",
            "cited_by"       : "Citations",
            "effect_size"    : "Effect size",
            "effect_type"    : "Type",
            "effect_duration": "Durée",
            "art_relevance"  : "Pertinence",
            "confidence"     : "Confiance",
            "note"           : "Note",
            "doi"            : "DOI",
        })
    )
else:
    print("Aucun article avec effect size extrait. Essayez de réduire MIN_CITATIONS.")

# ── Export Excel ──────────────────────────────────────────────────────────────
output_file = "resultats_moteur.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df.sort_values(["art_relevance", "confidence"], ascending=False).to_excel(
        writer, sheet_name="Tous les articles", index=False
    )
    top_articles.groupby("predictor").first().reset_index().to_excel(
        writer, sheet_name="Meilleur par prédicteur", index=False
    )

print(f"\n💾 Résultats exportés → {output_file}")


## 📊 Résultats — Early childhood support and parenting center
**Groupe cible :** Children aged 0-6 and their parents

| Métrique | Valeur |
|---|---|
| Prédicteurs explorés | 2 |
| Articles analysés | 8 |
| Articles avec effect size | 1 (12% si >0) |


### Prédicteurs identifiés

,Prédicteur,Domaine,Score,Justification
0,Family support,Extra-personnel,5,Le centre de soutien parental renforce directe...
1,Social integration,Extra-personnel,5,Les groupes de soutien entre pairs et l'espace...


### 📄 Articles avec effect size extrait

,Prédicteur,Titre,Année,Citations,Effect size,Type,Durée,Pertinence,Confiance,Note,DOI
6,Social integration,Effects of parenting education with expectant ...,2010,172,0.3,Cohen's d,None,4.0,4.0,Effet maintenu au suivi selon l'abstract,https://doi.org/10.1037/a0019691



💾 Résultats exportés → resultats_moteur.xlsx


## EXPORT FORMAT BOUSSOLE

In [12]:
# Mapping domaines FR → EN (format attendu par Boussole)
DOMAIN_FR_TO_EN = {
    'Environnement socio-politique':      'Socio-political environment',
    'Environnement matériel & éducation': 'Material environment and education',
    'Extra-personnel':                    'Extra-personal',
    'Intra-personnel':                    'Intra-personal',
    'Santé':                              'Health',
    'Travail & Activités':                'Work and Activities',
}

def make_boussole_df(df_src, project):
    """
    Format Boussole pour un sous-ensemble de df (avec ou sans effect size).
    Triés par prédicteur puis pertinence/confiance décroissante.

    Note (raw) :
      1. Passage exact de l'abstract d'où l'effect size a été extrait
      2. Note complémentaire du LLM
      3. Référence de l'article (titre, année, citations)
      4. Détails techniques (type, direction, confiance)
    """
    df_sorted = df_src.sort_values(
        ['predictor', 'art_relevance', 'confidence'],
        ascending=[True, False, False]
    ).reset_index(drop=True)

    tags_str = ', '.join(project.get('tags', []))
    rows = []
    for _, r in df_sorted.iterrows():
        domain_en = DOMAIN_FR_TO_EN.get(r.get('domain', ''), r.get('domain', ''))

        # ── Note (raw) — passage source de l'effect size ────────────────────
        note_parts = []

        # 1. Passage exact d'où l'effect size a été extrait
        if r.get('source_text'):
            note_parts.append(f'Extrait : "{r["source_text"]}"')

        # 2. Note complémentaire du LLM
        if r.get('note'):
            note_parts.append(r['note'])

        # 3. Référence article : titre + année + citations
        article_ref = r.get('title', '')
        if r.get('year'):     article_ref += f" ({r['year']})"
        if r.get('cited_by'): article_ref += f" — {r['cited_by']} citations"
        if article_ref:
            note_parts.append(f"Source : {article_ref}")

        # 4. Détails techniques
        details = []
        if r.get('effect_type'):      details.append(f"Type : {r['effect_type']}")
        if r.get('effect_direction'): details.append(f"Direction : {r['effect_direction']}")
        if r.get('confidence'):       details.append(f"Confiance LLM : {r['confidence']}/5")
        if details:
            note_parts.append(' | '.join(details))

        rows.append({
            'Project':             project.get('title', ''),
            'Project description': project.get('description', ''),
            'Tags':                tags_str,
            'Group':               project.get('target_group', ''),
            'Domain':              domain_en,
            'Predictor':           r['predictor'],
            'Effect size':         r['effect_size'],
            'Effect duration':     r.get('effect_duration', ''),
            'Note (raw)':          ' ; '.join(note_parts),
            'Article links':       r.get('doi', ''),
        })
    return pd.DataFrame(rows)

# Construction des deux DataFrames
boussole_with    = make_boussole_df(df[df['effect_size'].notna()], PROJECT)
boussole_without = make_boussole_df(df[df['effect_size'].isna()],  PROJECT)

display(Markdown(f'### 📋 Format Boussole — avec effect size ({len(boussole_with)} lignes)'))
display(boussole_with)

display(Markdown(f'### 📋 Format Boussole — sans effect size ({len(boussole_without)} lignes)'))
display(boussole_without)

output_file = 'resultats_moteur_boussole3.xlsx'
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Onglet 1 : articles avec effect size — prêt à injecter dans Boussole
    boussole_with.to_excel(writer, sheet_name='Boussole — avec effect size', index=False)
    # Onglet 2 : articles sans effect size — à compléter manuellement
    boussole_without.to_excel(writer, sheet_name='Boussole — sans effect size', index=False)
    # Onglet 3 : tous les articles avec colonnes techniques complètes
    df.sort_values(['art_relevance', 'confidence'], ascending=False).to_excel(
        writer, sheet_name='Tous les articles', index=False
    )

print(f'\n💾 Export : {output_file}')
print(f'   Onglet 1 : Boussole — avec effect size  ({len(boussole_with)} lignes)')
print(f'   Onglet 2 : Boussole — sans effect size  ({len(boussole_without)} lignes)')
print(f'   Onglet 3 : Tous les articles')
print(f'\n   {df["predictor"].nunique()} prédicteur(s) explorés au total')

### 📋 Format Boussole — avec effect size (1 lignes)

,Project,Project description,Tags,Group,Domain,Predictor,Effect size,Effect duration,Note (raw),Article links
0,Early childhood support and parenting center,\n Management of a welcoming space dedi...,"early childhood, parenting, child development,...",Children aged 0-6 and their parents,Extra-personal,Social integration,0.3,None,"Extrait : ""social development (d = .30)"" ; Eff...",https://doi.org/10.1037/a0019691


### 📋 Format Boussole — sans effect size (7 lignes)

,Project,Project description,Tags,Group,Domain,Predictor,Effect size,Effect duration,Note (raw),Article links
0,Early childhood support and parenting center,\n Management of a welcoming space dedi...,"early childhood, parenting, child development,...",Children aged 0-6 and their parents,Extra-personal,Family support,NaN,None,"Abstract trop général, pas d'effect size sur l...",https://doi.org/10.1093/schbul/sbt114
1,Early childhood support and parenting center,\n Management of a welcoming space dedi...,"early childhood, parenting, child development,...",Children aged 0-6 and their parents,Extra-personal,Family support,NaN,None,Méta-analyse qualitative sans données quantita...,https://doi.org/10.1093/jpepsy/jst020
2,Early childhood support and parenting center,\n Management of a welcoming space dedi...,"early childhood, parenting, child development,...",Children aged 0-6 and their parents,Extra-personal,Family support,NaN,None,Abstract non disponible - impossible d'extrair...,https://doi.org/10.1007/s11292-016-9256-0
3,Early childhood support and parenting center,\n Management of a welcoming space dedi...,"early childhood, parenting, child development,...",Children aged 0-6 and their parents,Extra-personal,Family support,NaN,None,Abstract non disponible - impossible d'extrair...,https://doi.org/10.1016/j.cpr.2020.101861
4,Early childhood support and parenting center,\n Management of a welcoming space dedi...,"early childhood, parenting, child development,...",Children aged 0-6 and their parents,Extra-personal,Social integration,NaN,None,"Extrait : ""Further, they differentiate interna...",https://doi.org/10.1080/15295192.2010.492040
5,Early childhood support and parenting center,\n Management of a welcoming space dedi...,"early childhood, parenting, child development,...",Children aged 0-6 and their parents,Extra-personal,Social integration,NaN,None,"Étude sur attitudes intergroupes, pas sur inté...",https://doi.org/10.1037/a0031436
6,Early childhood support and parenting center,\n Management of a welcoming space dedi...,"early childhood, parenting, child development,...",Children aged 0-6 and their parents,Extra-personal,Social integration,NaN,None,Article sur la gestion de la douleur périopéra...,https://doi.org/10.1097/aln.0b013e31823c1030



💾 Export : resultats_moteur_boussole3.xlsx
   Onglet 1 : Boussole — avec effect size  (1 lignes)
   Onglet 2 : Boussole — sans effect size  (7 lignes)
   Onglet 3 : Tous les articles

   2 prédicteur(s) explorés au total
